# Notebook 11: Observability & Evaluation with Langfuse

## Tracing, Prompt Management, Datasets & Scores for LangGraph

This notebook mirrors Notebook 10's production-observability arc, re-implemented with
[Langfuse](https://langfuse.com) — the open-source LLM engineering platform:

1. **Tracing** — automatic LangGraph tracing via the LangChain callback handler
2. **Trace organization** — names, tags, users, sessions, metadata
3. **Prompt management** — versioned prompts with labels, edited in the Langfuse UI
4. **Multi-agent & nested graphs** — supervisor workers and nested agents in one trace
5. **Evaluation** — datasets, experiments (heuristic + LLM-as-a-judge), A/B comparisons
6. **Scores & user feedback** — attach quality signals to live traces
7. **OpenTelemetry** — export raw OTEL spans straight to Langfuse (optional)

**Prerequisites**: Notebooks 01-10, a Langfuse project (cloud or self-hosted), and an `OPENAI_API_KEY`
**Time**: 30-40 minutes

Add to `.env` (see `.env.example`):

```bash
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com   # or us/jp region or your self-hosted URL
LANGFUSE_TRACING_ENVIRONMENT=development
```

---
## Setup: Langfuse Client & Configuration Check

`get_client()` reads `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_HOST` from the
environment. `auth_check()` verifies the credentials against the Langfuse API before we trace anything.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langfuse import get_client, Langfuse, propagate_attributes, observe

langfuse = get_client()

# ── Verify credentials ────────────────────────────────────────────────────────
LANGFUSE_READY = langfuse.auth_check()
print("=== Langfuse Configuration ===")
print(f"Host:        {os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")
print(f"Environment: {os.getenv('LANGFUSE_TRACING_ENVIRONMENT', 'not set')}")
print(f"Auth check:  {'✅ Authenticated' if LANGFUSE_READY else '❌ Failed — add LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY to .env'}")

if not LANGFUSE_READY:
    print("\nTo get credentials:")
    print("1. Create a project at https://cloud.langfuse.com (or your self-hosted instance)")
    print("2. Project Settings → API Keys → Create new keys")
    print("3. Add LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST to .env")

# ── Model used across this notebook ──────────────────────────────────────────
from langchain_openai import ChatOpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0)
print(f"\nLLM: {MODEL}")

=== Langfuse Configuration ===
Host:        https://cloud.langfuse.com
Environment: not set
Auth check:  ✅ Authenticated



LLM: gpt-4o-mini


---
## Setup: Demo Graph — LLM-Powered Support Ticket Agent

Throughout this notebook we use a small support-ticket graph: an LLM classifies the ticket
(category + priority via structured output), then drafts a reply. Every invocation becomes a
trace in Langfuse, so we can watch exactly what the model saw and produced.

In [2]:
from typing import Literal
from typing_extensions import TypedDict
from pydantic import BaseModel

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage


class TicketState(TypedDict):
    message: str
    category: str
    priority: str
    response: str


class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "feature", "general"]
    priority: Literal["low", "medium", "high"]


CLASSIFY_SYSTEM = SystemMessage(
    "You are a support ticket router. Classify the user's message into exactly one "
    "category (billing, technical, feature, general) and assign a priority "
    "(low, medium, high). Respond with structured output."
)

RESPOND_TEMPLATES = {
    "billing":   "Acknowledge the billing issue, apologize, and promise a resolution within 24 hours.",
    "technical": "Acknowledge the technical problem and promise an engineering update within 4 hours.",
    "feature":   "Thank the user for the suggestion and say it was added to the product roadmap.",
    "general":   "Acknowledge the question and promise a support response within 2 business days.",
}


def classify_ticket(state: TicketState) -> dict:
    structured_llm = llm.with_structured_output(TicketClassification)
    result = structured_llm.invoke([CLASSIFY_SYSTEM, HumanMessage(state["message"])])
    return {"category": result.category, "priority": result.priority}


def format_response(state: TicketState) -> dict:
    system = SystemMessage(
        f"You are a friendly support agent. {RESPOND_TEMPLATES[state['category']]} "
        "Keep the reply under 3 sentences."
    )
    reply = llm.invoke([system, HumanMessage(state["message"])])
    return {"response": reply.content}


builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("respond", format_response)
builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

app = builder.compile(checkpointer=MemorySaver())

# ── Smoke test ────────────────────────────────────────────────────────────────
_smoke = app.invoke(
    {"message": "I was charged twice this month", "category": "", "priority": "", "response": ""},
    {"configurable": {"thread_id": "setup-verify-001"}},
)
print("✅ Demo graph compiled and verified")
print(f"   Input:    'I was charged twice this month'")
print(f"   Category: {_smoke['category']} | Priority: {_smoke['priority']}")
print(f"   Response: {_smoke['response']}")

✅ Demo graph compiled and verified
   Input:    'I was charged twice this month'
   Category: billing | Priority: high
   Response: I apologize for the inconvenience caused by the double charge. We will investigate this billing issue and ensure it is resolved within the next 24 hours. Thank you for your patience!


---
# Part 1: Tracing Fundamentals

Langfuse traces LangGraph through the LangChain callback handler — pass `CallbackHandler` via
`config={"callbacks": [...]}` on any `.invoke()` / `.stream()` call. We progress from a raw
automatic trace to fully named, tagged, and organized traces.

## Example 1: Basic Trace

The minimal setup. The handler captures the graph run, both LLM calls (classify + respond),
their prompts, completions, token usage, and cost — no other code changes.

In [3]:
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()

print("Running graph — open your Langfuse project Traces view to see the result\n")
for chunk in app.stream(
    {"message": "I was charged twice this month", "category": "", "priority": "", "response": ""},
    config={"callbacks": [langfuse_handler], "configurable": {"thread_id": "ex1-basic"}},
    stream_mode="updates",
):
    print(chunk)

Running graph — open your Langfuse project Traces view to see the result



{'classify': {'category': 'billing', 'priority': 'high'}}


{'respond': {'response': 'I apologize for the inconvenience caused by the double charge. We will investigate this billing issue and ensure it is resolved within the next 24 hours. Thank you for your patience!'}}


## Example 2: Named & Filterable Traces

Production traces need names, tags, user/session identifiers, and metadata so you can find and
filter them in the dashboard. Wrap the invocation in `propagate_attributes()` and set `run_name`
in the config.

In [4]:
with propagate_attributes(
    trace_name="Ticket Agent - Billing Query",
    user_id="user-42",
    session_id="session-abc",
    tags=["production", "v2.0", "customer-support"],
    metadata={"environment": "production", "feature_flags": ["new_routing_v2"]},
):
    result = app.invoke(
        {"message": "I was charged twice this month", "category": "", "priority": "", "response": ""},
        config={"callbacks": [langfuse_handler], "run_name": "handle-ticket-billing-001", "configurable": {"thread_id": "ex2-001"}},
    )

print(result["response"])
print("\n→ In Langfuse, filter the Traces table by Trace Name = 'Ticket Agent - Billing Query'")
print("  and by tags 'production' / 'customer-support'.")

I apologize for the inconvenience caused by the double charge. We will investigate this billing issue and ensure it is resolved within the next 24 hours. Thank you for your patience!

→ In Langfuse, filter the Traces table by Trace Name = 'Ticket Agent - Billing Query'
  and by tags 'production' / 'customer-support'.


## Example 3: Sessions & Users

Pass the same `session_id` across multiple related invocations to group them into a **session**
in Langfuse (user journey view), and a stable `user_id` to see all activity of one user across
sessions.

In [5]:
# Three related turns in one support conversation
conversation = [
    "Hi, I was charged twice this month",
    "Actually it happened on my yearly plan, can you check that?",
    "Thanks! Also, is there a dark mode?",
]

with propagate_attributes(
    trace_name="Ticket Agent - Multi-turn Session",
    user_id="user-42",
    session_id="support-session-001",
    tags=["customer-support", "multi-turn"],
    metadata={"channel": "web-chat"},
):
    for i, message in enumerate(conversation):
        turn = app.invoke(
            {"message": message, "category": "", "priority": "", "response": ""},
            config={"callbacks": [langfuse_handler], "run_name": f"session-turn-{i+1}", "configurable": {"thread_id": f"ex3-turn-{i+1}"}},
        )
        print(f"[turn {i+1}] {turn['category']:<10} → {turn['response'][:80]}")

print("\n→ Open Langfuse → Sessions and look up 'support-session-001' to see the full journey.")

[turn 1] billing    → I apologize for the inconvenience caused by the double charge. We will investiga


[turn 2] billing    → I apologize for the billing issue you're experiencing with your yearly plan. I w


[turn 3] feature    → Thank you for your suggestion! I'm happy to let you know that dark mode has been

→ Open Langfuse → Sessions and look up 'support-session-001' to see the full journey.


## Example 4: Manual Instrumentation with `@observe()`

The callback handler covers LangChain/LangGraph. For your own Python functions, use the
`@observe()` decorator. `propagate_attributes` only sets trace attributes — it starts no span —
so to nest custom observations under one root we wrap the workflow in an explicit
`start_as_current_observation` span; everything inside (including the graph run) attaches to it.

In [6]:
import time


@observe()
def sanitize_message(message: str) -> str:
    """Custom preprocessing step — appears as its own observation in the trace."""
    time.sleep(0.1)  # simulate real work
    return message.strip()


@observe()
def audit_log(category: str, priority: str) -> None:
    """Business-logic side effect we also want visibility on."""
    print(f"   [audit] ticket categorized as {category}/{priority}")


with langfuse.start_as_current_observation(as_type="span", name="ticket-pipeline") as pipeline:
    with propagate_attributes(
        trace_name="Ticket Agent - With Custom Observations",
        session_id="support-session-001",
        tags=["customer-support", "custom-instrumentation"],
    ):
        cleaned = sanitize_message("   The app keeps crashing on iOS 17   ")
        pipeline.update(input=cleaned)
        result = app.invoke(
            {"message": cleaned, "category": "", "priority": "", "response": ""},
            config={"callbacks": [langfuse_handler], "run_name": "observed-ticket-run", "configurable": {"thread_id": "ex4-001"}},
        )
        audit_log(result["category"], result["priority"])
        pipeline.update(output=result["response"])

print(f"\nResponse: {result['response']}")
print("→ 'ticket-pipeline' is the trace root; the graph run and both @observe functions nest under it.")

   [audit] ticket categorized as technical/high

Response: Thank you for bringing this issue to our attention. We understand how frustrating this can be and are currently investigating the app crashes on iOS 17. You can expect an engineering update within the next 4 hours.
→ 'ticket-pipeline' is the trace root; the graph run and both @observe functions nest under it.


---
# Part 2: Prompt Management

Langfuse Prompt Management is a versioned Prompt CMS: prompts live in Langfuse, are fetched by
name at runtime, and can be edited, versioned, and promoted with labels (e.g. `production`) —
without redeploying code.

## Create & Version a Managed Prompt

We create a system prompt for the response node, then publish an improved **v2** and promote it
to `production` with a label. `{{variables}}` are compiled at runtime.

In [7]:
# Create only on first run — re-running the notebook must not duplicate versions
try:
    _existing = langfuse.get_prompt("ticket-respond-system")
    print(f"Prompt 'ticket-respond-system' already exists (v{_existing.version}) — skipping creation.")
except Exception:
    # v1 — initial prompt, promoted to production
    langfuse.create_prompt(
        name="ticket-respond-system",
        prompt=(
            "You are a friendly support agent for a SaaS product. "
            "Ticket category: {{category}}, priority: {{priority}}. "
            "Guidance: {{guidance}} Keep the reply under 3 sentences."
        ),
        labels=["production"],
    )

    # v2 — improved wording, also promoted to production (labels are movable pointers)
    langfuse.create_prompt(
        name="ticket-respond-system",
        prompt=(
            "You are a friendly, empathetic support agent for a SaaS product. "
            "Ticket category: {{category}}, priority: {{priority}}. "
            "Guidance: {{guidance}} Keep the reply under 3 sentences. "
            "Always thank the user and state the expected response time explicitly."
        ),
        labels=["production"],
    )

# Fetch the current production version and compile its variables
prompt = langfuse.get_prompt("ticket-respond-system")
compiled = prompt.compile(
    category="billing",
    priority="high",
    guidance="Acknowledge the issue, apologize, and promise a resolution within 24 hours.",
)
print(f"Version:  {prompt.version}")
print(f"Labels:   {prompt.labels}")
print(f"Compiled:\n{compiled}")

Prompt 'ticket-respond-system' already exists (v4) — skipping creation.
Version:  4
Labels:   ['production', 'latest']
Compiled:
You are a friendly, empathetic support agent for a SaaS product. Ticket category: billing, priority: high. Guidance: Acknowledge the issue, apologize, and promise a resolution within 24 hours. Keep the reply under 3 sentences. Always thank the user and state the expected response time explicitly.


## Use the Managed Prompt in the Graph

The respond node now pulls the production prompt from Langfuse at runtime. Each LLM generation
is automatically **linked to the prompt name and version** — so you can compare output quality
across prompt versions in the dashboard.

In [8]:
def format_response_managed(state: TicketState) -> dict:
    prompt = langfuse.get_prompt("ticket-respond-system")
    system = SystemMessage(prompt.compile(
        category=state["category"],
        priority=state["priority"],
        guidance=RESPOND_TEMPLATES[state["category"]],
    ))
    reply = llm.invoke([system, HumanMessage(state["message"])])
    return {"response": reply.content}


managed_builder = StateGraph(TicketState)
managed_builder.add_node("classify", classify_ticket)
managed_builder.add_node("respond", format_response_managed)
managed_builder.add_edge(START, "classify")
managed_builder.add_edge("classify", "respond")
managed_builder.add_edge("respond", END)
app_managed = managed_builder.compile(checkpointer=MemorySaver())

with propagate_attributes(
    trace_name="Ticket Agent - Managed Prompt",
    tags=["customer-support", "prompt-management"],
    metadata={"prompt_name": "ticket-respond-system"},
):
    result = app_managed.invoke(
        {"message": "My invoice is wrong and support never answers", "category": "", "priority": "", "response": ""},
        config={"callbacks": [langfuse_handler], "run_name": "managed-prompt-run", "configurable": {"thread_id": "pm-001"}},
    )

print(result["response"])
print("\n→ In the trace, open the generation and see the linked prompt name + version.")

I’m really sorry to hear about the issue with your invoice and the delay in support. I assure you that we will resolve this within the next 24 hours. Thank you for your patience!

→ In the trace, open the generation and see the linked prompt name + version.


---
# Part 3: Multi-Agent Systems

A supervisor routes work to specialized agents. Langfuse captures the supervisor decision, each
worker agent, and every tool call as nested observations within one trace.

## Supervisor + Worker Agents

We reuse `langgraph-supervisor` (from Notebook 09) with two tool-calling agents — a billing
specialist and a technical specialist — each with its own tool.

In [9]:
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor


@tool
def lookup_billing_policy(query: str) -> str:
    """Look up the billing and refund policy."""
    return ("Billing policy: refunds are processed within 5-7 business days. "
            "Duplicate charges are reversed within 24 hours.")


@tool
def lookup_status_page(query: str) -> str:
    """Look up known incidents and system status."""
    return ("Status: all systems operational. No known incidents affecting "
            "authentication or checkout right now.")


billing_agent = create_react_agent(
    model=llm, tools=[lookup_billing_policy], name="billing_specialist",
    prompt="You handle billing questions. Use the billing policy tool, then answer in 2 sentences.",
)
tech_agent = create_react_agent(
    model=llm, tools=[lookup_status_page], name="tech_specialist",
    prompt="You handle technical issues. Use the status tool, then answer in 2 sentences.",
)

supervisor_builder = create_supervisor(
    [billing_agent, tech_agent],
    model=llm,
    prompt=(
        "Route each support message to billing_specialist or tech_specialist. "
        "Reply with FINISH when the user's question has been answered."
    ),
    output_mode="last_message",
)
multi_agent = supervisor_builder.compile()

with propagate_attributes(
    trace_name="Multi-Agent - Supervisor",
    session_id="support-session-001",
    tags=["customer-support", "multi-agent"],
):
    for chunk in multi_agent.stream(
        {"messages": [{"role": "user", "content": "I was double charged, is there a known outage too?"}]},
        config={"callbacks": [langfuse_handler], "run_name": "supervisor-run"},
        stream_mode="updates",
    ):
        print(chunk)

/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_68036/3655897657.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  billing_agent = create_react_agent(
/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_68036/3655897657.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  tech_agent = create_react_agent(


{'supervisor': {'messages': [HumanMessage(content='I was double charged, is there a known outage too?', additional_kwargs={}, response_metadata={}, id='a0680db6-de3e-4392-8c53-461a059509b9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 104, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_15014e7912', 'id': 'chatcmpl-EmD013oznNzSiBGecWj2IpWOq3rx7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--01a1f7c7-aef8-78f3-8ef9-c1c5f36b5f4d-0', tool_calls=[{'name': 'transfer_to_billing_specialist', 'args': {}, 'id': 'call_TWCvUsSbEP9jdj7oBjWuhs9U', 'type': 'tool_call'}], invalid

{'billing_specialist': {'messages': [AIMessage(content='If you were double charged, the duplicate charges should be reversed within 24 hours. There are currently no known outages affecting billing.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 225, 'total_tokens': 252, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5632fb73e4', 'id': 'chatcmpl-EmD03d33XjPzujZBGlQK09zCuSIkf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, name='billing_specialist', id='lc_run--01a1f7c7-b70d-79d0-a5d4-a07b8dfea496-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 225, 'output_tokens': 27, 'total_tokens': 252, 'input_token_details': {'audio'

{'supervisor': {'messages': [HumanMessage(content='I was double charged, is there a known outage too?', additional_kwargs={}, response_metadata={}, id='a0680db6-de3e-4392-8c53-461a059509b9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 104, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_15014e7912', 'id': 'chatcmpl-EmD013oznNzSiBGecWj2IpWOq3rx7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--01a1f7c7-aef8-78f3-8ef9-c1c5f36b5f4d-0', tool_calls=[{'name': 'transfer_to_billing_specialist', 'args': {}, 'id': 'call_TWCvUsSbEP9jdj7oBjWuhs9U', 'type': 'tool_call'}], invalid

## Nested Agents in One Trace

When one agent calls another (agent-as-tool), use a predefined trace ID and
`start_as_current_observation(trace_context=...)` to merge both graphs into a **single trace**.

In [10]:
from langchain_core.messages import HumanMessage

# One trace ID shared by the main agent and the nested sub-agent
nested_trace_id = Langfuse.create_trace_id()
print(f"Trace ID: {nested_trace_id}")

# Sub-agent: a small research graph
sub_builder = StateGraph(TicketState)
sub_builder.add_node("classify", classify_ticket)
sub_builder.add_edge(START, "classify")
sub_builder.add_edge("classify", END)
sub_agent = sub_builder.compile(checkpointer=MemorySaver())


@tool
def classify_sub_agent(message: str) -> str:
    """Delegate ticket classification to a specialized sub-agent."""
    with langfuse.start_as_current_observation(
        as_type="span", name="call-classifier-sub-agent",
        trace_context={"trace_id": nested_trace_id},
    ) as span:
        span.update(input=message)
        response = sub_agent.invoke(
            {"message": message, "category": "", "priority": "", "response": ""},
            config={"callbacks": [langfuse_handler], "run_name": "classifier-sub-agent", "configurable": {"thread_id": "nested-sub-001"}},
        )
        span.update(output=response["category"])
    return f"{response['category']} ({response['priority']} priority)"


main_agent = create_react_agent(
    model=llm, tools=[classify_sub_agent], name="main_agent",
    prompt="You are a triage coordinator. Use the classify_sub_agent tool to categorize tickets.",
)

with langfuse.start_as_current_observation(
    as_type="span", name="run-nested-agent",
    trace_context={"trace_id": nested_trace_id},
) as span:
    with propagate_attributes(
        trace_name="Nested Agent - Main + Sub",
        tags=["customer-support", "nested-agent"],
    ):
        span.update(input="Login is broken on mobile")
        response = main_agent.invoke(
            {"messages": [HumanMessage("Login is broken on mobile")]},
            config={"callbacks": [langfuse_handler], "run_name": "main-agent"},
        )
        span.update(output=response["messages"][-1].content)

print(response["messages"][-1].content)
print(f"\n→ One trace ({nested_trace_id}) contains main agent, tool call, and sub-agent runs.")

Trace ID: 03f954e2cc97a890dba8ca16b2d9952f


/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_68036/4224607175.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  main_agent = create_react_agent(


The ticket has been classified as a technical issue with high priority.

→ One trace (03f954e2cc97a890dba8ca16b2d9952f) contains main agent, tool call, and sub-agent runs.


---
# Part 4: Evaluation — Datasets & Experiments

**Datasets** hold test cases (input + expected output). **Experiments** run your app over a
dataset and attach scores, so every prompt/model/graph change becomes a comparable experiment
run in Langfuse. This mirrors Notebook 10's four eval examples: accuracy, completeness,
tone — plus a real A/B comparison persisted in the platform.

## Create a Dataset

Ticket classification test cases, exactly the style of Notebook 10's `TEST_CASES` — but now
versioned and reusable in the Langfuse UI.

In [11]:
TEST_CASES = [
    ("I need a refund for my subscription", "billing",   "high"),
    ("The app keeps crashing on iOS 17",    "technical", "high"),
    ("Would love to see dark mode added",   "feature",   "low"),
    ("How do I reset my password?",         "general",   "medium"),
    ("I was charged twice this month",      "billing",   "high"),
    ("Login button is broken on mobile",    "technical", "high"),
]

# Create the dataset and items only on first run (items are matched by input message)
try:
    _existing = langfuse.get_dataset("ticket-classification")
    existing_inputs = {it.input["message"] for it in _existing.items}
except Exception:
    existing_inputs = set()
    langfuse.create_dataset(
        name="ticket-classification",
        description="Support ticket classification benchmark (mirrors Notebook 10 test cases)",
        metadata={"author": "langgraph-tutorials", "type": "benchmark"},
    )

for message, category, priority in TEST_CASES:
    if message in existing_inputs:
        continue
    langfuse.create_dataset_item(
        dataset_name="ticket-classification",
        input={"message": message},
        expected_output={"category": category, "priority": priority},
    )

dataset = langfuse.get_dataset("ticket-classification")
print(f"Dataset '{dataset.name}': {len(dataset.items)} items")
for item in dataset.items:
    print(f"  {item.input['message']:<45} → {item.expected_output['category']}")

Dataset 'ticket-classification': 6 items
  Login button is broken on mobile              → technical
  I was charged twice this month                → billing
  How do I reset my password?                   → general
  Would love to see dark mode added             → feature
  The app keeps crashing on iOS 17              → technical
  I need a refund for my subscription           → billing


## Evaluators: Accuracy, Completeness, Tone

Three heuristic evaluators, ported from Notebook 10. Each receives the task output and the
dataset item, and returns one or more scores.

In [12]:
REQUIRED_KEYWORDS = {
    "billing":   ["refund", "charge", "24 hours", "billing"],
    "technical": ["engineering", "update", "hours", "status"],
    "feature":   ["thank", "roadmap"],
    "general":   ["support", "days"],
}


def eval_accuracy(*, output, expected_output, **kwargs):
    predicted = output["category"]
    expected = expected_output["category"]
    return {"name": "accuracy", "value": 1.0 if predicted == expected else 0.0,
            "comment": f"predicted={predicted!r} expected={expected!r}"}


def eval_completeness(*, output, **kwargs):
    required = REQUIRED_KEYWORDS.get(output["category"], [])
    response = output["response"].lower()
    hits = sum(1 for kw in required if kw in response)
    value = hits / len(required) if required else 1.0
    return {"name": "completeness", "value": round(value, 2),
            "comment": f"keywords={hits}/{len(required)}"}


def eval_tone(*, output, **kwargs):
    resp = output["response"].lower()
    polite = sum(1 for p in ["thank", "please", "sorry", "will", "team"] if p in resp)
    cold = sum(1 for p in ["cannot", "impossible", "never", "refuse"] if p in resp)
    raw = min(1.0, polite / 2) * (0.5 if cold else 1.0)
    return {"name": "tone", "value": round(raw, 2),
            "comment": "professional" if raw >= 0.5 else "needs work"}


print("Evaluators defined: accuracy, completeness, tone")

Evaluators defined: accuracy, completeness, tone


## Run Experiments — A/B Compare Two Prompts

The task invokes the graph on each dataset item; experiments `classifier-prompt-v1` and
`classifier-prompt-v2` differ only in the classifier system prompt. Compare average scores in
Langfuse → Datasets → ticket-classification → Experiments.

In [13]:
from langchain_core.messages import SystemMessage as SM, HumanMessage as HM

CLASSIFY_V2 = SM(
    "You are a support ticket router. Classify into exactly one category: "
    "billing (charges, refunds, money, subscriptions, invoices), "
    "technical (crashes, errors, bugs, slowness, outages), "
    "feature (suggestions, wishes, improvements), or general (anything else). "
    "Assign priority high/medium/low based on user impact."
)


def run_experiment_for_prompt(name: str, system_message) -> None:
    def task(*, item, **kwargs):
        structured_llm = llm.with_structured_output(TicketClassification)
        cls = structured_llm.invoke([system_message, HM(item.input["message"])])
        reply = llm.invoke([
            SM(f"You are a friendly support agent. The ticket is {cls.category} ({cls.priority}). "
               "Write a 2-sentence acknowledgment with the expected response time."),
            HM(item.input["message"]),
        ])
        return {"category": cls.category, "priority": cls.priority, "response": reply.content}

    experiment = dataset.run_experiment(
        name=name,
        description=f"Classifier comparison — {name}",
        task=task,
        evaluators=[eval_accuracy, eval_completeness, eval_tone],
        max_concurrency=3,
    )
    # Local summary so the notebook is readable without opening the dashboard
    for r in experiment.item_results:
        scores = {e.name: e.value for e in r.evaluations}
        print(f"  {r.item.input['message'][:45]:<45} acc={scores.get('accuracy')} tone={scores.get('tone')}")


print("=== Experiment: classifier-prompt-v1 (original prompt) ===")
run_experiment_for_prompt("classifier-prompt-v1", CLASSIFY_SYSTEM)
print("\n=== Experiment: classifier-prompt-v2 (expanded definitions) ===")
run_experiment_for_prompt("classifier-prompt-v2", CLASSIFY_V2)

=== Experiment: classifier-prompt-v1 (original prompt) ===


  Login button is broken on mobile              acc=1.0 tone=1.0
  I was charged twice this month                acc=1.0 tone=1.0
  How do I reset my password?                   acc=1.0 tone=1.0
  Would love to see dark mode added             acc=1.0 tone=1.0
  The app keeps crashing on iOS 17              acc=1.0 tone=0.5
  I need a refund for my subscription           acc=1.0 tone=1.0

=== Experiment: classifier-prompt-v2 (expanded definitions) ===


  Login button is broken on mobile              acc=1.0 tone=1.0
  I was charged twice this month                acc=1.0 tone=1.0
  How do I reset my password?                   acc=1.0 tone=0.5
  Would love to see dark mode added             acc=1.0 tone=1.0
  The app keeps crashing on iOS 17              acc=1.0 tone=1.0
  I need a refund for my subscription           acc=1.0 tone=1.0


## LLM-as-a-Judge

Heuristics only go so far. An evaluator can itself call an LLM — here a judge grades
helpfulness 1-5 with a rubric. Judge scores land next to the heuristic ones on the same
experiment runs.

In [14]:
import re


def eval_judge_helpfulness(*, input, output, **kwargs):
    judge = ChatOpenAI(model=MODEL, temperature=0)
    grading = judge.invoke([
        SM("Grade the helpfulness of this support reply on a 1-5 scale. "
           "Reply with exactly: GRADE: <int>"),
        HM(f"User message: {input['message']}\nReply: {output['response']}"),
    ])
    match = re.search(r"GRADE:\s*(\d+)", grading.content)
    grade = int(match.group(1)) if match else 0
    return {"name": "judge_helpfulness", "value": grade / 5,
            "comment": grading.content[:120]}


def judge_task(*, item, **kwargs):
    return app_managed.invoke(
        {"message": item.input["message"], "category": "", "priority": "", "response": ""},
        {"configurable": {"thread_id": f"judge-{abs(hash(item.input['message'])) % 9999}"}},
    )


experiment = dataset.run_experiment(
    name="managed-prompt-with-judge",
    description="Managed-prompt agent graded by an LLM judge",
    task=judge_task,
    evaluators=[eval_accuracy, eval_judge_helpfulness],
    max_concurrency=3,
)
print(f"Experiment '{experiment.name}' finished — open Datasets → ticket-classification → Experiments.")

Experiment 'managed-prompt-with-judge' finished — open Datasets → ticket-classification → Experiments.


---
# Part 5: Scores & User Feedback

Scores attach quality signals to a specific trace. Use `create_score` for post-hoc scoring
(human review, CI checks), or `score_current_trace` right inside the running code.

## Attach Scores to a Live Trace

Run the graph under a trace ID we control, then score it: numeric accuracy, categorical
quality, and a user-feedback style thumbs up/down with a comment.

In [15]:
feedback_trace_id = Langfuse.create_trace_id()

with langfuse.start_as_current_observation(
    as_type="span", name="scored-ticket-run",
    trace_context={"trace_id": feedback_trace_id},
) as span:
    with propagate_attributes(
        trace_name="Ticket Agent - Scored Run",
        user_id="user-42",
        session_id="support-session-001",
        tags=["customer-support", "scored"],
    ):
        span.update(input="I want my money back")
        result = app.invoke(
            {"message": "I want my money back", "category": "", "priority": "", "response": ""},
            config={"callbacks": [langfuse_handler], "run_name": "scored-run-001", "configurable": {"thread_id": "scored-001"}},
        )
        span.update(output=result["response"])

# 1) Numeric score — automated check
correct = result["category"] == "billing"
langfuse.create_score(
    trace_id=feedback_trace_id, name="classification-correct",
    value=1.0 if correct else 0.0, data_type="NUMERIC",
    comment=f"predicted={result['category']}",
)

# 2) Categorical score — human review bucket
langfuse.create_score(
    trace_id=feedback_trace_id, name="human-review",
    value="good", data_type="CATEGORICAL", comment="Response time stated, tone fine.",
)

# 3) Numeric user feedback (thumbs up)
langfuse.create_score(
    trace_id=feedback_trace_id, name="user-feedback",
    value=1, data_type="NUMERIC", comment="User clicked thumbs up in the chat UI.",
)

print(f"Scored trace {feedback_trace_id}")
print(f"  classification-correct: {correct}")
print(f"  user-feedback: 1 (thumbs up)")
print(f"  human-review: 'good'")
print("\n→ In the trace detail view, all three scores appear in the Scores panel.")
print("  Aggregate them later via Langfuse dashboards / Metrics API.")

Scored trace 59f5cb1118ac6e9ccd13881a73fa5d05
  classification-correct: True
  user-feedback: 1 (thumbs up)
  human-review: 'good'

→ In the trace detail view, all three scores appear in the Scores panel.
  Aggregate them later via Langfuse dashboards / Metrics API.


## Score Inside the Running Code

`score_current_trace` attaches a score from within the instrumented execution itself — useful
for runtime quality gates (e.g. self-consistency checks) without a second API call.

In [16]:
def respond_with_self_check(state: TicketState) -> dict:
    output = format_response(state)
    # Runtime quality gate: response must mention a time frame
    has_timeframe = any(w in output["response"].lower() for w in ["hour", "day"])
    langfuse.score_current_trace(
        name="self-check-timeframe", value=1.0 if has_timeframe else 0.0,
        comment="States an expected response time",
    )
    return output


checked_builder = StateGraph(TicketState)
checked_builder.add_node("classify", classify_ticket)
checked_builder.add_node("respond", respond_with_self_check)
checked_builder.add_edge(START, "classify")
checked_builder.add_edge("classify", "respond")
checked_builder.add_edge("respond", END)
app_checked = checked_builder.compile(checkpointer=MemorySaver())

with propagate_attributes(trace_name="Ticket Agent - Runtime Self-Check", tags=["scored"]):
    r = app_checked.invoke(
        {"message": "The export feature is broken", "category": "", "priority": "", "response": ""},
        config={"callbacks": [langfuse_handler], "run_name": "self-check-run", "configurable": {"thread_id": "selfcheck-001"}},
    )
print(r["response"])
print("\n→ 'self-check-timeframe' score was attached from inside the node.")

Thank you for bringing this to our attention! We appreciate your feedback, and I’ve added the issue with the export feature to our product roadmap for further investigation. If you have any more details to share, feel free to let us know!

→ 'self-check-timeframe' score was attached from inside the node.


---
# Part 6 (Optional): OpenTelemetry-Native Export

Langfuse is an OTLP backend: any OpenTelemetry-instrumented code can export spans directly to
`{LANGFUSE_HOST}/api/public/otel` — no Langfuse SDK required. Useful for non-Python services or
when you already have an OTEL pipeline. Spans carrying `langfuse.*` attributes map onto the
Langfuse data model (trace name, input/output, metadata).

In [17]:
import base64
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com").rstrip("/")
public_key = os.getenv("LANGFUSE_PUBLIC_KEY", "")
secret_key = os.getenv("LANGFUSE_SECRET_KEY", "")

if LANGFUSE_READY:
    auth = base64.b64encode(f"{public_key}:{secret_key}".encode()).decode()
    provider = TracerProvider()
    provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(
        endpoint=f"{host}/api/public/otel/v1/traces",
        headers={"Authorization": f"Basic {auth}", "x-langfuse-ingestion-version": "4"},
    )))
    tracer = trace.get_tracer("langfuse-otel-demo")

    with tracer.start_as_current_span("support-summary-job") as span:
        span.set_attribute("langfuse.trace.name", "OTEL - Support Summary Job")
        span.set_attribute("langfuse.trace.tags", '["otel", "batch-job"]')
        span.set_attribute("langfuse.trace.metadata.job_type", "nightly-summary")
        span.set_attribute("langfuse.observation.input", '{"date": "2026-11-20"}')
        summary = "42 tickets processed: 18 billing, 14 technical, 6 feature, 4 general."
        span.set_attribute("langfuse.observation.output", summary)
        print(summary)

    provider.shutdown()  # flush the batch processor
    print("\n→ Check Langfuse Traces for 'OTEL - Support Summary Job'.")
else:
    print("Skipped — add Langfuse credentials to .env to run the OTEL example.")

42 tickets processed: 18 billing, 14 technical, 6 feature, 4 general.

→ Check Langfuse Traces for 'OTEL - Support Summary Job'.


---
## Summary

| Capability | Langfuse API | Where to see it |
|---|---|---|
| Automatic LangGraph tracing | `CallbackHandler` in `config["callbacks"]` | Traces |
| Trace organization | `propagate_attributes(...)` + `run_name` | Traces (filterable) |
| Users & sessions | `user_id`, `session_id` attributes | Users / Sessions views |
| Custom instrumentation | `@observe()`, `start_as_current_observation` | Trace detail (nested spans) |
| Prompt management | `create_prompt` / `get_prompt` / labels | Prompts; linked on generations |
| Multi-agent / nested graphs | handler + shared `trace_id` | Single merged trace |
| Evaluation | `create_dataset` + `dataset.run_experiment` | Datasets → Experiments |
| LLM-as-a-judge | evaluator function calling an LLM | Experiment scores |
| Scores & feedback | `create_score`, `score_current_trace` | Trace Scores panel, dashboards |
| OTEL-native export | OTLP to `/api/public/otel` | Traces |

**Next steps**

- [Langfuse Documentation](https://langfuse.com/docs)
- [LangGraph Integration Cookbook](https://langfuse.com/integrations/frameworks/langgraph)
- [Datasets & Experiments](https://langfuse.com/docs/datasets/overview)
- [OpenTelemetry Integration](https://langfuse.com/docs/opentelemetry/get-started)
- Langfuse is open source: [github.com/langfuse/langfuse](https://github.com/langfuse/langfuse)

In [18]:
# Flush all buffered events to Langfuse before the process exits
try:
    langfuse.flush()
    print("✅ All Langfuse events flushed.")
except Exception as e:
    print(f"⚠️ Flush failed (check credentials/network): {e}")

✅ All Langfuse events flushed.
